In [139]:
import pandas as pd
import os
import spacy
from spacy import displacy
from spacy.matcher import PhraseMatcher
from spacy.util import filter_spans

In [ ]:
telerama = pd.read_csv("/Users/pol/Downloads/telerama_clean.csv")

aliases = pd.read_csv("/Users/pol/Downloads/aliases.csv")  # or however you import
aliases = aliases.dropna(subset=["mbz_alias"])

alias_to_id = dict(
    zip(aliases["mbz_alias"], aliases["dz_name"])
)

In [137]:
def keep_alias(alias):
    tokens = alias.split()
    
    if len(tokens) > 1:
        return True
    return len(alias) >= 5

alias_to_id = {
    alias: artist_id
    for alias, artist_id in alias_to_id.items()
    if keep_alias(alias)
}


In [138]:
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser", "tagger"])

In [140]:
matcher = PhraseMatcher(nlp.vocab)

patterns = [nlp.make_doc(alias) for alias in alias_to_id.keys()]
matcher.add("ARTIST", patterns)

In [143]:
telerama.head()

,source_name,issue,DocHeader,article_title,article_author,article_text,article_annex,date,article_page,section,reviewed_score,article_type,reviewed_artist,reviewed_title,source_origin,url
0,Télérama,2802,Disques,Serge Reggiani - Succès & confidences,Anne-Marie Paquotte,"Une compilation de plus chez un éditeur qui, i...",Serge Reggiani / Succès & confidences > 2 CD P...,2003-09-27,p. 63,Disques,2F,review,Serge Reggiani,Succès & confidences,Europresse,NaN
1,Télérama,2802,Disques,"Désolés, Warren",Hugo Cassavetti,"C'est une maigre consolation, mais Warren Zevo...",Warren Zevon / I'll sleep when I'm dead > anth...,2003-09-27,p. 65,Disques,NaN,NaN,Warren Zevon,I'll sleep when I'm dead,Europresse,NaN
2,Télérama,2802,Disques,Adam Green - Friends of mine,François Gorin,Dire qu'on a passé l'été l'oreille scotchée au...,Adam Green / Friends of mine > 1 CD Pias. 3F,2003-09-27,p. 63,Disques,3F,review,Adam Green,Friends of mine,Europresse,NaN
3,Télérama,2802,Disques,Cesaria Evora - Voz d'amor,Eliane Azoulay,"Dès les premières notes de guitare, on se retr...",Cesaria Evora / Voz d'amor > 1 CD BMG. 3F,2003-09-27,p. 64,Disques,3F,review,Cesaria Evora,Voz d'amor,Europresse,NaN
4,Télérama,2802,Disques,Herman Düne - Mas cambios et Mash Concrete Met...,François Gorin,Les deux frères de Herman Düne jouent et chant...,Herman Düne / Mas cambios > 1 CD Track & Field...,2003-09-27,p. 64,Disques,3F,review,Herman Düne,Mas cambios,Europresse,NaN


In [ ]:
df = telerama[1:100]

results = []

for doc, article_id in zip(nlp.pipe(df["article_text"]), df["reviewed_artist"]):
    matches = matcher(doc)
    
    spans = [doc[start:end] for _, start, end in matches]
    spans = filter_spans(spans)  # keeps longest, removes overlaps
    
    for span in spans:
        results.append({
            "article_id": article_id,
            "matched_string": span.text,
            "artist_id": alias_to_id.get(span.text)
        })

/Users/pol/Documents/GitHub/music_artists_data_paper/.venv/lib/python3.13/site-packages/spacy/pipeline/lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


ValueError: not enough values to unpack (expected 2, got 1)

In [145]:
matches_df = pd.DataFrame(results)
matches_df

,article_id,matched_string,artist_id
0,Warren Zevon,Warren Zevon,Warren Zevon
1,Warren Zevon,Roland,Roland Van Campenhout
2,Warren Zevon,Thompson,Marko Perković Thompson
3,Warren Zevon,Elton John,Elton John
4,Warren Zevon,Jackson Browne,Jackson Browne
...,...,...,...
678,Stephanie McKay,Portishead,Portishead
679,Stephanie McKay,Burt Bacharach,Burt Bacharach
680,Stephanie McKay,Dionne Warwick,Dionne Warwick
681,Batata y su rumba palenquera,Harmonia,Harmonia


In [148]:
[row for row in matches_df[:100]]

['article_id', 'matched_string', 'artist_id']

In [150]:
t = matches_df.groupby(["article_id", "artist_id"]).size()
t

article_id    artist_id       
Adam Green    Charles Trenet      1
              Grupo Green         1
              Jonathan Richman    1
              Lou Reed            1
              The Kills           1
                                 ..
Yann Tiersen  Poulain             1
              Prefab Sprout       1
              Virgin              1
              Yann Tiersen        2
Zen Zila      Aurelio Voltaire    1
Length: 473, dtype: int64

article_id                                artist_id       
Adam Green - Friends of mine              1                   1
                                          Charles Trenet      1
                                          Dire                1
                                          Grupo Green         1
                                          JE                  1
                                                             ..
Zen Zila - 2 pull-overs, 1 vieux costard  Aurelio Voltaire    1
                                          LA                  1
                                          Ulysse              1
                                          bice                1
                                          et_                 2
Length: 1238, dtype: int64